# 🩺 Diabetic Retinopathy Grading — Complete Pipeline
**Target:** 95%+ Accuracy | 5-Class Classification (DR Levels 0–4)  
**Backbone:** EfficientNet-B4 (pretrained) | **Framework:** PyTorch + timm + Albumentations

---
| Level | Grade | Count | % |
|---|---|---|---|
| 0 | No DR | 25,802 | 73.5% |
| 1 | Mild DR | 2,438 | 6.9% |
| 2 | Moderate DR | 5,288 | 15.1% |
| 3 | Severe DR | 872 | 2.5% |
| 4 | Proliferative DR | 708 | 2.0% |

---
### Folder layout expected
```
project/
├── labels.csv
├── data/
│   └── train/
│       ├── 10_left.jpeg
│       ├── 10_right.jpeg
│       └── ...
└── DR_notebook.ipynb   ← this file
```

## Cell 1 — Install Dependencies

In [ ]:
# Run once — restart kernel after installation
!pip install torch torchvision timm opencv-python-headless \
             albumentations scikit-learn pandas numpy \
             matplotlib seaborn tqdm --quiet
print("✅ Dependencies installed")

## Cell 2 — Imports

In [ ]:
import os, cv2, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, accuracy_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

warnings.filterwarnings("ignore")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"timm     : {timm.__version__}")

## Cell 3 — Configuration
> ✏️ **Edit these paths to match your setup before running anything else.**

In [ ]:
class Config:
    # ── Paths ─────────────────────────────────────────────────────────────────
    CSV_PATH       = "labels.csv"        # your label CSV
    TRAIN_IMG_DIR  = "data/train"        # retinal fundus images folder
    TEST_IMG_DIR   = "data/test"         # test images (optional)
    IMG_EXT        = ".jpeg"             # change to ".png" if needed
    CHECKPOINT_DIR = "checkpoints"       # where best weights are saved

    # ── Model ─────────────────────────────────────────────────────────────────
    MODEL_NAME  = "efficientnet_b4"      # EfficientNet-B4 backbone
    NUM_CLASSES = 5                      # DR grades 0-4
    IMG_SIZE    = 448                    # input resolution

    # ── Training ──────────────────────────────────────────────────────────────
    EPOCHS      = 40
    BATCH_SIZE  = 16                     # reduce to 8 if GPU OOM
    LR          = 1e-4
    WEIGHT_DECAY= 1e-4
    GRAD_CLIP   = 1.0
    NUM_WORKERS = 4                      # set 0 on Windows if errors
    SEED        = 42
    VAL_SPLIT   = 0.15
    EARLY_STOP  = 7                      # patience epochs

    # ── Class Balancing ───────────────────────────────────────────────────────
    USE_WEIGHTED_SAMPLER = True          # oversample minority classes
    USE_CLASS_WEIGHTS    = True          # weighted loss function
    MIXUP_ALPHA          = 0.3           # MixUp strength (0 = disabled)

    # ── Inference ─────────────────────────────────────────────────────────────
    TTA_STEPS      = 5                   # test-time augmentation rounds
    CONF_THRESHOLD = 0.70                # below this → flagged as uncertain

    # ── Device ────────────────────────────────────────────────────────────────
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = Config()
os.makedirs(CFG.CHECKPOINT_DIR, exist_ok=True)

print(f"Device         : {CFG.DEVICE}")
print(f"Image size     : {CFG.IMG_SIZE}x{CFG.IMG_SIZE}")
print(f"Batch size     : {CFG.BATCH_SIZE}")
print(f"Epochs         : {CFG.EPOCHS}")
print(f"Checkpoint dir : {CFG.CHECKPOINT_DIR}/")

## Cell 4 — Seed & Label Map

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(CFG.SEED)

LABEL_NAMES = {
    0: "No DR",
    1: "Mild DR",
    2: "Moderate DR",
    3: "Severe DR",
    4: "Proliferative DR",
}
print("✅ Seed set:", CFG.SEED)
print("Label map  :", LABEL_NAMES)

## Cell 5 — Exploratory Data Analysis
Visualise the class imbalance — this drives every balancing decision.

In [ ]:
df = pd.read_csv(CFG.CSV_PATH)
print(f"Total samples : {len(df)}")
print(f"Columns       : {list(df.columns)}")
print(df.head(8))

# ── Class distribution ───────────────────────────────────────────────────────
vc = df["level"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart
axes[0].bar(
    [f"L{i}\n{LABEL_NAMES[i]}" for i in vc.index],
    vc.values,
    color=["#2196F3","#4CAF50","#FF9800","#F44336","#9C27B0"]
)
axes[0].set_title("Class Distribution (Count)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Samples")
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 100, str(v), ha="center", fontsize=10)

# Pie chart
axes[1].pie(
    vc.values,
    labels=[f"L{i}: {LABEL_NAMES[i]}" for i in vc.index],
    autopct="%1.1f%%",
    colors=["#2196F3","#4CAF50","#FF9800","#F44336","#9C27B0"],
    startangle=140,
)
axes[1].set_title("Class Distribution (Proportion)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()
print("⚠️  Severe imbalance: Level 0 is 73.5% — balancing is critical!")

## Cell 6 — Ben Graham Preprocessing
Subtracts a Gaussian-blurred copy of each image from itself.  
This removes background lighting gradients and dramatically enhances  
micro-features: microaneurysms, exudates, haemorrhages.

In [ ]:
def ben_graham_preprocess(img: np.ndarray, img_size: int) -> np.ndarray:
    """
    1. Resize to (img_size x img_size)
    2. Subtract Gaussian-blurred version to remove lighting gradient
    3. Clip to valid [0, 255] range
    """
    img     = cv2.resize(img, (img_size, img_size))
    blurred = cv2.GaussianBlur(img, (0, 0), img_size // 30)
    img     = cv2.addWeighted(img, 4, blurred, -4, 128)
    img     = np.clip(img, 0, 255).astype(np.uint8)
    return img


def load_image(img_path: str, img_size: int = CFG.IMG_SIZE) -> np.ndarray:
    """Load → BGR-to-RGB → Ben Graham → return RGB uint8 array."""
    img = cv2.imread(img_path)
    if img is None:
        return np.zeros((img_size, img_size, 3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = ben_graham_preprocess(img, img_size)
    return img


# ── Quick visual demo (requires at least one image in TRAIN_IMG_DIR) ─────────
sample_row = df.sample(1).iloc[0]
sample_path = os.path.join(CFG.TRAIN_IMG_DIR, sample_row["image"] + CFG.IMG_EXT)

if os.path.exists(sample_path):
    raw = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
    raw = cv2.resize(raw, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    processed = ben_graham_preprocess(raw.copy(), CFG.IMG_SIZE)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(raw);       axes[0].set_title("Original",   fontsize=12); axes[0].axis("off")
    axes[1].imshow(processed); axes[1].set_title("Ben Graham", fontsize=12); axes[1].axis("off")
    plt.suptitle(f"DR Level {sample_row['level']} — {LABEL_NAMES[sample_row['level']]}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print(f"⚠️  Sample image not found at {sample_path}")
    print("   Preprocessing functions are defined and ready.")

print("✅ Ben Graham preprocessing defined")

## Cell 7 — Augmentation Pipelines
Three pipelines: **train** (heavy), **validation** (none), **TTA** (light random).

In [ ]:
def get_train_transforms(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    """Heavy augmentation for training — forces the model to generalise."""
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                           rotate_limit=30, p=0.5),
        # Geometric distortions (simulate camera/lens variation)
        A.OneOf([
            A.GridDistortion(p=1.0),
            A.ElasticTransform(p=1.0),
            A.OpticalDistortion(p=1.0),
        ], p=0.3),
        # Contrast / sharpness
        A.OneOf([
            A.CLAHE(clip_limit=2.0, p=1.0),
            A.Sharpen(p=1.0),
            A.Emboss(p=1.0),
        ], p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2,
                      saturation=0.2, hue=0.1, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(p=0.2),
        # CoarseDropout: hide random patches (like CutOut / RandomErasing)
        A.CoarseDropout(max_holes=8,
                        max_height=img_size // 10,
                        max_width=img_size // 10, p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    """Validation: only normalise — no random augmentation."""
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_tta_transforms(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    """Light stochastic augmentation for test-time augmentation."""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

print("✅ Augmentation pipelines defined (train / val / TTA)")

## Cell 8 — PyTorch Dataset

In [ ]:
class DRDataset(Dataset):
    """
    Diabetic Retinopathy Dataset.

    Parameters
    ----------
    df        : DataFrame with columns ['image', 'level']
    img_dir   : directory containing the image files
    transform : Albumentations Compose pipeline
    img_ext   : file extension, default '.jpeg'
    """
    def __init__(self, df: pd.DataFrame, img_dir: str,
                 transform=None, img_ext: str = CFG.IMG_EXT):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.img_ext   = img_ext

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image"] + self.img_ext)
        image    = load_image(img_path, CFG.IMG_SIZE)   # Ben Graham applied
        label    = int(row["level"])

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label

print("✅ DRDataset class defined")

## Cell 9 — Class Balancing Utilities
Three complementary mechanisms to fight the 73.5% vs 2% imbalance:
1. `WeightedRandomSampler` — oversamples Level 3 & 4 during training
2. `compute_class_weights` — penalises mistakes on rare classes more
3. MixUp (Cell 11) — blends samples across class boundaries

In [ ]:
def compute_class_weights(labels: np.ndarray,
                          num_classes: int = CFG.NUM_CLASSES) -> torch.Tensor:
    """
    Inverse-frequency class weights → passed to the loss function.
    Rare classes get higher weight so errors on them hurt more.
    """
    counts  = np.bincount(labels, minlength=num_classes).astype(float)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * num_classes   # normalise: mean ≈ 1
    return torch.tensor(weights, dtype=torch.float32)


def build_weighted_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    """
    WeightedRandomSampler: each class gets equal expected frequency per epoch.
    Minority classes (Level 3, 4) are oversampled; majority (Level 0) is
    undersampled — WITHOUT discarding any data.
    """
    counts        = Counter(labels)
    class_weight  = {cls: 1.0 / cnt for cls, cnt in counts.items()}
    sample_weights = [class_weight[lbl] for lbl in labels]
    return WeightedRandomSampler(
        weights     = sample_weights,
        num_samples = len(sample_weights),
        replacement = True,
    )


# ── Verify weights ────────────────────────────────────────────────────────────
w = compute_class_weights(df["level"].values)
print("Class weights (higher = rarer class):")
for i, wt in enumerate(w):
    print(f"  Level {i} ({LABEL_NAMES[i]:<18s}): {wt:.4f}")

## Cell 10 — Train/Val Split & DataLoaders

In [ ]:
# Stratified split → preserves class ratios in both sets
train_df, val_df = train_test_split(
    df,
    test_size    = CFG.VAL_SPLIT,
    stratify     = df["level"],
    random_state = CFG.SEED,
)
print(f"Train : {len(train_df)} samples")
print(f"Val   : {len(val_df)} samples")

# ── Datasets ──────────────────────────────────────────────────────────────────
train_ds = DRDataset(train_df, CFG.TRAIN_IMG_DIR, get_train_transforms())
val_ds   = DRDataset(val_df,   CFG.TRAIN_IMG_DIR, get_val_transforms())

# ── Weighted sampler for training ─────────────────────────────────────────────
sampler = None
shuffle = True
if CFG.USE_WEIGHTED_SAMPLER:
    sampler = build_weighted_sampler(train_df["level"].values)
    shuffle = False    # sampler & shuffle are mutually exclusive in PyTorch

train_loader = DataLoader(
    train_ds,
    batch_size  = CFG.BATCH_SIZE,
    sampler     = sampler,
    shuffle     = shuffle,
    num_workers = CFG.NUM_WORKERS,
    pin_memory  = True,
    drop_last   = True,        # drop last incomplete batch for stable BN
)
val_loader = DataLoader(
    val_ds,
    batch_size  = CFG.BATCH_SIZE * 2,   # no gradients → can fit more
    shuffle     = False,
    num_workers = CFG.NUM_WORKERS,
    pin_memory  = True,
)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")

# ── Sanity check one batch ────────────────────────────────────────────────────
imgs, lbls = next(iter(train_loader))
print(f"Batch image shape : {imgs.shape}")   # (B, 3, 448, 448)
print(f"Batch labels      : {lbls.tolist()}")

## Cell 11 — MixUp Augmentation
Interpolates two samples in pixel-space: `mixed = λ·x_i + (1-λ)·x_j`.  
Forces the model to learn smooth decision boundaries between grades.

In [ ]:
def mixup_data(x: torch.Tensor, y: torch.Tensor,
               alpha: float = 0.3):
    """Return mixed inputs, both label tensors, and lambda."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Loss = λ · L(pred, y_a) + (1-λ) · L(pred, y_b)."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print("✅ MixUp functions defined (alpha =", CFG.MIXUP_ALPHA, ")")

## Cell 12 — Model Architecture
EfficientNet-B4 backbone (pretrained on ImageNet) + custom 2-layer head.  
The two-stage head with GELU improves probability calibration on medical grading.

In [ ]:
class DRClassifier(nn.Module):
    """
    EfficientNet-B4 + custom classification head:
      GlobalAvgPool → Dropout(0.4) → FC(features→512) → GELU
      → Dropout(0.2) → FC(512→num_classes)
    """
    def __init__(self,
                 model_name : str = CFG.MODEL_NAME,
                 num_classes: int = CFG.NUM_CLASSES,
                 pretrained : bool = True):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained  = pretrained,
            num_classes = 0,       # remove built-in head
            global_pool = "avg",
        )
        num_features = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(num_features, 512),
            nn.GELU(),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.backbone(x))


model = DRClassifier().to(CFG.DEVICE)
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone       : {CFG.MODEL_NAME}")
print(f"Total params   : {total:,}")
print(f"Trainable      : {trainable:,}")
print(f"Device         : {CFG.DEVICE}")

## Cell 13 — Loss Function, Optimiser & Scheduler
- **LabelSmoothingLoss** with class weights: smooths targets from {0,1} and penalises rare-class errors more  
- **AdamW**: weight-decoupled Adam  
- **CosineAnnealingLR**: decays LR smoothly from `1e-4` → `1e-6` over 40 epochs

In [ ]:
class LabelSmoothingLoss(nn.Module):
    """
    Cross-entropy with label smoothing + optional class weights.
    smoothing=0.1 → target confidence = 0.9 (not 1.0), which
    prevents overconfidence on the dominant Level 0 class.
    """
    def __init__(self, num_classes: int, smoothing: float = 0.1,
                 weight: torch.Tensor = None):
        super().__init__()
        self.smoothing   = smoothing
        self.num_classes = num_classes
        self.weight      = weight

    def forward(self, pred: torch.Tensor,
                target: torch.Tensor) -> torch.Tensor:
        confidence = 1.0 - self.smoothing
        smooth_val = self.smoothing / (self.num_classes - 1)

        one_hot = torch.zeros_like(pred).scatter_(
            1, target.unsqueeze(1), 1)
        smooth_oh = one_hot * confidence + (1 - one_hot) * smooth_val

        log_prob = F.log_softmax(pred, dim=1)

        if self.weight is not None:
            w    = self.weight.to(pred.device)[target]
            loss = -(smooth_oh * log_prob).sum(dim=1)
            return (loss * w).mean()
        return -(smooth_oh * log_prob).sum(dim=1).mean()


# ── Class weights ─────────────────────────────────────────────────────────────
class_weights = None
if CFG.USE_CLASS_WEIGHTS:
    class_weights = compute_class_weights(df["level"].values).to(CFG.DEVICE)
    print("Class weights:", class_weights.cpu().numpy().round(3))

criterion = LabelSmoothingLoss(
    CFG.NUM_CLASSES, smoothing=0.1, weight=class_weights)

# ── Optimiser ─────────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

# ── Cosine LR scheduler ───────────────────────────────────────────────────────
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)

# ── Mixed-precision scaler ────────────────────────────────────────────────────
scaler = GradScaler()

print("✅ Loss / Optimiser / Scheduler ready")

## Cell 14 — Metrics
**Quadratic Weighted Kappa (QWK)** is the primary metric for DR grading.  
Unlike accuracy, QWK penalises large ordinal mistakes (e.g. predicting Level 0 when true is Level 4) more than small ones.

In [ ]:
def quadratic_weighted_kappa(y_true, y_pred,
                             n: int = CFG.NUM_CLASSES) -> float:
    """
    Cohen's Kappa with quadratic weighting.
    Range [-1, 1]:  0 = random, 1 = perfect, >0.81 = clinically strong.
    """
    return cohen_kappa_score(
        y_true, y_pred,
        weights="quadratic",
        labels=list(range(n)),
    )

print("✅ QWK metric defined")
# Demo
demo_true = [0, 0, 1, 2, 3, 4]
demo_pred = [0, 1, 1, 2, 2, 4]
print(f"Demo QWK (near-perfect): {quadratic_weighted_kappa(demo_true, demo_pred):.4f}")

## Cell 15 — Training & Validation Loop Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion,
                    scaler, scheduler=None, mixup_alpha=0.0):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, labels in pbar:
        images = images.to(CFG.DEVICE, non_blocking=True)
        labels = labels.to(CFG.DEVICE, non_blocking=True)

        # ── Optional MixUp ────────────────────────────────────────────────────
        if mixup_alpha > 0:
            images, labels_a, labels_b, lam = mixup_data(
                images, labels, mixup_alpha)

        optimizer.zero_grad()

        # ── Forward pass (AMP) ────────────────────────────────────────────────
        with autocast():
            logits = model(images)
            if mixup_alpha > 0:
                loss = mixup_criterion(
                    criterion, logits, labels_a, labels_b, lam)
            else:
                loss = criterion(logits, labels)

        # ── Backward + gradient clip ──────────────────────────────────────────
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        # ── Stats ─────────────────────────────────────────────────────────────
        preds = logits.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    if scheduler:
        scheduler.step()

    return (total_loss / total,
            correct / total,
            quadratic_weighted_kappa(all_labels, all_preds))


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc="  Val  ", leave=False):
        images = images.to(CFG.DEVICE, non_blocking=True)
        labels = labels.to(CFG.DEVICE, non_blocking=True)

        with autocast():
            logits = model(images)
            loss   = criterion(logits, labels)

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (total_loss / total,
            correct / total,
            quadratic_weighted_kappa(all_labels, all_preds),
            np.array(all_labels),
            np.array(all_preds))

print("✅ Training & validation functions defined")

## Cell 16 — Run Training
Trains for up to 40 epochs with:
- Cosine LR decay
- Early stopping (patience = 7)
- Checkpoint saved whenever validation QWK improves

In [ ]:
history    = {k: [] for k in
              ["train_loss","val_loss","train_acc","val_acc","train_qwk","val_qwk"]}
best_qwk   = -1.0
no_improve = 0
v_true = v_pred = None   # will hold last-epoch val predictions for reporting

print(f"{'='*75}")
print(f"  Training on {CFG.DEVICE} | {len(train_loader)} batches/epoch")
print(f"{'='*75}\n")

for epoch in range(1, CFG.EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc, tr_qwk = train_one_epoch(
        model, train_loader, optimizer, criterion,
        scaler, scheduler, CFG.MIXUP_ALPHA
    )
    vl_loss, vl_acc, vl_qwk, v_true, v_pred = validate(
        model, val_loader, criterion
    )

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)
    history["train_qwk"].append(tr_qwk)
    history["val_qwk"].append(vl_qwk)

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]

    print(
        f"Ep {epoch:03d}/{CFG.EPOCHS} | LR {lr_now:.1e} | "
        f"TrL {tr_loss:.4f} TrA {tr_acc:.4f} TrQWK {tr_qwk:.4f} | "
        f"VlL {vl_loss:.4f} VlA {vl_acc:.4f} VlQWK {vl_qwk:.4f} | "
        f"{elapsed:.0f}s"
    )

    if vl_qwk > best_qwk:
        best_qwk   = vl_qwk
        no_improve = 0
        ckpt_path  = os.path.join(CFG.CHECKPOINT_DIR, "best_model.pth")
        torch.save({
            "epoch"     : epoch,
            "state_dict": model.state_dict(),
            "val_acc"   : vl_acc,
            "val_qwk"   : vl_qwk,
            "config"    : vars(CFG),
        }, ckpt_path)
        print(f"  ★ Checkpoint saved  (best QWK = {best_qwk:.4f})")
    else:
        no_improve += 1
        if no_improve >= CFG.EARLY_STOP:
            print(f"\n  Early stopping at epoch {epoch}.")
            break

print(f"\n✅ Training complete | Best QWK = {best_qwk:.4f}")

## Cell 17 — Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training History", fontsize=14, fontweight="bold")

metrics = [
    ("Accuracy",  "train_acc",  "val_acc",  "steelblue"),
    ("Loss",      "train_loss", "val_loss", "darkorange"),
    ("QWK",       "train_qwk",  "val_qwk",  "seagreen"),
]
for ax, (title, tr_key, vl_key, color) in zip(axes, metrics):
    ax.plot(history[tr_key], label="Train", color=color, linewidth=2)
    ax.plot(history[vl_key], label="Val",   color=color,
            linestyle="--", linewidth=2)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150)
plt.show()
print("Saved → training_history.png")

## Cell 18 — Confusion Matrix & Classification Report

In [ ]:
cm     = confusion_matrix(v_true, v_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
names  = [f"L{i}\n{LABEL_NAMES[i]}" for i in range(CFG.NUM_CLASSES)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in zip(
    axes,
    [cm, cm_pct],
    ["d", ".1f"],
    ["Count", "Row %"],
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=names, yticklabels=names, ax=ax,
                linewidths=0.5)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True",      fontsize=11)
    ax.set_title(f"Confusion Matrix ({title})", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print("\nClassification Report:")
print(classification_report(v_true, v_pred,
                             target_names=list(LABEL_NAMES.values())))
print(f"Final Accuracy : {accuracy_score(v_true, v_pred):.4f}")
print(f"Final QWK      : {quadratic_weighted_kappa(v_true, v_pred):.4f}")

## Cell 19 — Load Best Checkpoint for Inference

In [ ]:
CKPT_PATH = os.path.join(CFG.CHECKPOINT_DIR, "best_model.pth")
ckpt      = torch.load(CKPT_PATH, map_location=CFG.DEVICE)

best_model = DRClassifier().to(CFG.DEVICE)
best_model.load_state_dict(ckpt["state_dict"])
best_model.eval()

print(f"Loaded checkpoint from epoch : {ckpt['epoch']}")
print(f"Saved val accuracy           : {ckpt['val_acc']:.4f}")
print(f"Saved val QWK                : {ckpt['val_qwk']:.4f}")
print("✅ best_model ready for inference")

## Cell 20 — Test-Time Augmentation (TTA) Prediction
Runs the model `N` times on lightly-augmented versions of the same image  
and averages the softmax probabilities. Adds ~1–2% accuracy at zero extra training cost.

In [ ]:
@torch.no_grad()
def tta_predict(model: nn.Module,
                image_np: np.ndarray,
                n_steps: int = CFG.TTA_STEPS) -> tuple:
    """
    Parameters
    ----------
    model    : trained DRClassifier (in eval mode)
    image_np : RGB numpy array, uint8, already Ben-Graham-preprocessed
    n_steps  : number of TTA rounds

    Returns
    -------
    (predicted_class, confidence, probability_array)
    """
    model.eval()
    tta_tf      = get_tta_transforms()
    probs_accum = np.zeros(CFG.NUM_CLASSES)

    for _ in range(n_steps):
        aug  = tta_tf(image=image_np)["image"].unsqueeze(0).to(CFG.DEVICE)
        out  = model(aug)
        prob = F.softmax(out, dim=1).cpu().numpy()[0]
        probs_accum += prob

    probs_accum /= n_steps
    pred_class   = int(probs_accum.argmax())
    confidence   = float(probs_accum.max())
    return pred_class, confidence, probs_accum

print(f"✅ tta_predict defined  (TTA steps = {CFG.TTA_STEPS})")

## Cell 21 — Single Image Inference

In [ ]:
def infer_single_image(img_path: str, model: nn.Module) -> dict:
    """
    Full inference pipeline for one image file.
    Returns grade, label, confidence, uncertainty flag,
    and per-class probabilities.
    """
    img_np = load_image(img_path, CFG.IMG_SIZE)
    pred_class, confidence, probs = tta_predict(model, img_np)

    return {
        "image"        : img_path,
        "grade"        : pred_class,
        "label"        : LABEL_NAMES[pred_class],
        "confidence"   : confidence,
        "uncertain"    : confidence < CFG.CONF_THRESHOLD,
        "probabilities": {LABEL_NAMES[i]: float(probs[i])
                          for i in range(CFG.NUM_CLASSES)},
    }


def display_prediction(result: dict):
    """Pretty-print inference result with probability bars."""
    print(f"\n{'─'*55}")
    print(f"  Image      : {result['image']}")
    print(f"  Grade      : {result['grade']}  —  {result['label']}")
    print(f"  Confidence : {result['confidence']:.2%}")
    print(f"  Status     : {'⚠️  UNCERTAIN' if result['uncertain'] else '✅ CONFIDENT'}")
    print(f"\n  Per-class probabilities:")
    for name, prob in result["probabilities"].items():
        bar  = "█" * int(prob * 40)
        print(f"    {name:<20s} {prob:.4f}  {bar}")
    print(f"{'─'*55}\n")


# ── Test on one validation image ──────────────────────────────────────────────
sample     = val_df.sample(1, random_state=7).iloc[0]
sample_path = os.path.join(CFG.TRAIN_IMG_DIR, sample["image"] + CFG.IMG_EXT)

if os.path.exists(sample_path):
    result = infer_single_image(sample_path, best_model)
    display_prediction(result)
    print(f"  True label : {sample['level']} — {LABEL_NAMES[sample['level']]}")
else:
    print(f"⚠️  Image not found: {sample_path}")
    print("   Set sample_path to any retinal image to test.")

## Cell 22 — Visualise Predictions on Sample Images

In [ ]:
def visualise_predictions(model, df, img_dir, n=8):
    """Grid of sample images with true vs predicted labels."""
    samples = df.sample(n, random_state=99).reset_index(drop=True)
    fig, axes = plt.subplots(2, n // 2, figsize=(20, 8))
    axes = axes.flatten()

    for i, (_, row) in enumerate(samples.iterrows()):
        img_path = os.path.join(img_dir, row["image"] + CFG.IMG_EXT)
        if not os.path.exists(img_path):
            axes[i].axis("off")
            continue

        img_np = load_image(img_path, CFG.IMG_SIZE)
        pred_class, conf, _ = tta_predict(model, img_np, n_steps=3)

        true_lbl = LABEL_NAMES[row["level"]]
        pred_lbl = LABEL_NAMES[pred_class]
        color    = "green" if pred_class == row["level"] else "red"

        axes[i].imshow(img_np)
        axes[i].set_title(
            f"True: L{row['level']} {true_lbl}\nPred: L{pred_class} {pred_lbl} ({conf:.0%})",
            fontsize=9, color=color, fontweight="bold"
        )
        axes[i].axis("off")

    plt.suptitle("Sample Predictions  (Green = Correct, Red = Wrong)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("sample_predictions.png", dpi=150)
    plt.show()


visualise_predictions(best_model, val_df, CFG.TRAIN_IMG_DIR, n=8)

## Cell 23 — Real-Time Camera / Fundus Camera Inference
Runs inference on every frame from a webcam or USB fundus camera.  
Predictions, confidence, and per-class probability bars are overlaid.  
Press **`q`** to quit.

In [ ]:
def realtime_camera_inference(model: nn.Module, camera_id: int = 0):
    """
    Live DR grading from camera feed.
    Press 'q' to exit.
    """
    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        print("❌ Cannot open camera. Check camera_id or connection.")
        return

    print("📷 Real-time DR inference started — press 'q' to quit.")
    model.eval()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # ── Preprocess frame ──────────────────────────────────────────────────
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img_pp  = ben_graham_preprocess(img_rgb, CFG.IMG_SIZE)

        pred_class, confidence, probs = tta_predict(model, img_pp, n_steps=3)

        label     = LABEL_NAMES[pred_class]
        uncertain = confidence < CFG.CONF_THRESHOLD
        bar_color = (0, 165, 255) if uncertain else (0, 200, 0)

        # ── Overlay ───────────────────────────────────────────────────────────
        disp    = cv2.resize(frame, (820, 620))
        overlay = disp.copy()
        cv2.rectangle(overlay, (0, 0), (420, 190), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.55, disp, 0.45, 0, disp)

        status = "UNCERTAIN ⚠" if uncertain else "CONFIDENT ✓"
        cv2.putText(disp, f"Grade {pred_class}: {label}",
                    (10, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255,255,255), 2)
        cv2.putText(disp, f"Conf: {confidence:.1%}  [{status}]",
                    (10, 68), cv2.FONT_HERSHEY_SIMPLEX, 0.70, bar_color, 2)

        y0 = 100
        for i, (name, p) in enumerate(zip(LABEL_NAMES.values(), probs)):
            bar_w = int(p * 200)
            cv2.rectangle(disp,
                          (10,      y0 + i * 17),
                          (10 + bar_w, y0 + i * 17 + 13),
                          bar_color if i == pred_class else (70, 70, 70), -1)
            cv2.putText(disp, f"L{i}: {p:.3f}  {name[:8]}",
                        (220, y0 + i * 17 + 11),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)

        cv2.imshow("DR Real-Time Inference  |  press q to quit", disp)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("📷 Camera closed.")


# ── To run: ───────────────────────────────────────────────────────────────────
# realtime_camera_inference(best_model, camera_id=0)
print("✅ realtime_camera_inference defined")
print("   Call: realtime_camera_inference(best_model, camera_id=0)")

## Cell 24 — Summary & Next Steps

### What was done
| Step | Detail |
|---|---|
| **Preprocessing** | Ben Graham (Gaussian subtraction) applied to every image |
| **Balancing** | WeightedRandomSampler + class-weighted LabelSmoothingLoss + MixUp |
| **Model** | EfficientNet-B4 pretrained + 2-layer GELU head |
| **Training** | AdamW + CosineAnnealingLR + GradientClipping + AMP |
| **Evaluation** | Accuracy + Quadratic Weighted Kappa + Confusion Matrix |
| **Inference** | TTA (5-step) + confidence threshold flagging |
| **Real-time** | Live camera overlay with per-class probability bars |

### To push past 95%
1. **Ensemble** 2–3 seeds or `efficientnet_b5` + `convnext_base`
2. **Progressive resizing**: train at 256 → 320 → 448
3. **Grad-CAM visualisation** to verify the model attends to the macula
4. **Pseudo-labeling**: add high-confidence test predictions back into training